# BTC/USD 1-Minute Data — Complete Advanced Analytics Pipeline

**Single notebook combining all 8 pipeline stages** (01 Load & Quality → 08 Dashboard).

**Dataset:** Kaggle — Bitcoin Historical Data (`btcusd_1-min_data.csv`)
https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data

In [ ]:
# inline rendering for notebooks
%matplotlib inline
from IPython.display import display

---

# Stage 01 — Load & Data Quality Audit

Reads the raw 393 MB CSV with memory-efficient dtypes, audits missing/duplicate/gap/plausibility issues, saves the cleaned 1-minute parquet and `audit_01.json`.

In [ ]:
import pandas as pd
import numpy as np
import json, time

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

t0 = time.time()
df = pd.read_csv(r"C:\Users\Admin\Downloads\archive\btcusd_1-min_data.csv",
                 dtype={'Open':'float32','High':'float32','Low':'float32','Close':'float32','Volume':'float32'},
                 nrows=None)
print("load seconds:", round(time.time()-t0,1))
print("shape:", df.shape)
print("memory MB:", round(df.memory_usage(deep=True).sum()/1e6,1))

df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s', errors='coerce')
df = df.sort_values('Timestamp').reset_index(drop=True)


In [ ]:
# ---- Data Quality Audit ----
audit = {}

# missing
miss = df.isna().sum()
miss_pct = (df.isna().mean()*100).round(4)
# Pre-2017 known: High/Low imputed (this dataset has some high=low=c close artifacts? just report)
audit['missing'] = {c:int(miss[c]) for c in miss.index}
audit['missing_pct'] = {c:float(miss_pct[c]) for c in miss_pct.index}

# duplicates: exact rows
audit['duplicate_exact_rows'] = int(df.duplicated().sum())

# duplicate timestamps
audit['duplicate_timestamps'] = int(df['Timestamp'].duplicated().sum())

# date range
audit['first_ts'] = str(df['Timestamp'].min())
audit['last_ts'] = str(df['Timestamp'].max())

# expected minute count vs actual
full_range = pd.date_range(df['Timestamp'].min(), df['Timestamp'].max(), freq='min')
audit['expected_minutes'] = int(len(full_range))
audit['actual_minutes'] = int(len(df))
audit['missing_gap_minutes'] = int(len(full_range)-len(df))

# impossible values / sanity
audit['const_rows'] = int((df['High']==df['Low']).sum())
o = df['Open'].astype(float); h = df['High'].astype(float); l = df['Low'].astype(float); c = df['Close'].astype(float)
audit['rows_o_not_between_l_h'] = int(((o < l) | (o > h)).sum())
audit['rows_c_not_between_l_h'] = int(((c < l) | (c > h)).sum())
audit['negative_prices'] = int((o<0).sum() | (h<0).sum() | (l<0).sum() | (c<0).sum())
audit['negative_volume'] = int((df['Volume']<0).sum())
audit['zero_volume_minutes'] = int((df['Volume']==0).sum())
audit['zero_volume_pct'] = round(100*(df['Volume']==0).mean(),3)
audit['zero_price_minutes'] = int((c==0).sum())

# zero-volume by year
df['year'] = df['Timestamp'].dt.year
zv_by_year = df[df['Volume']==0].groupby('year').size()
tot_by_year = df.groupby('year').size()
audit['zero_volume_pct_by_year'] = {str(i): round(100*zv_by_year.get(i,0)/tot_by_year.get(i,1),2) for i in sorted(df['year'].unique())}

# Constant / near-zero variance columns
for col in ['Open','High','Low','Close','Volume']:
    s = df[col].astype(float)
    audit[f'var_{col}'] = float(s.var())
    audit[f'std_{col}'] = float(s.std())
audit['unique_prices_total'] = int(df['Close'].nunique())

# price ranges
audit['min_price'] = float(df['Close'].min())
audit['max_price'] = float(df['Close'].max())
audit['max_high'] = float(df['High'].max())
audit['min_low'] = float(df['Low'].min())
audit['max_volume'] = float(df['Volume'].max())
audit['mean_volume'] = float(df['Volume'].mean())
audit['pct_prices_penny'] = float(round(100*(df['Close'].astype(float) < 1).mean(),4))  # prices < $1 era

# biggest 1-min return spikes (rough, log returns)
px = df['Close'].astype(float).to_numpy()
logr = np.diff(np.log(px + 1e-12))
df_tmp = df.iloc[1:].copy()
df_tmp['logret'] = logr
audit['max_1min_logret'] = float(logr.max())
audit['min_1min_logret'] = float(logr.min())

# store cleaned parquet (float32, dedupe timestamps, drop year col artefact? keep)
df.to_parquet(r"C:\Users\Admin\Downloads\archive\out\btc_1min_clean.parquet", index=False)
audit['saved_rows'] = int(len(df))

with open(r"C:\Users\Admin\Downloads\archive\out\audit_01.json",'w') as f:
    json.dump(audit, f, indent=2, default=str)
print(json.dumps(audit, indent=2, default=str))

---

## 02 - Daily Dataset Build & Feature Engineering

Aggregates 1-minute bars into daily/monthly/yearly OHLCV and engineers leakage-safe features (returns, rolling volatility, drawdown, calendar). Saves parquet + summary_daily.json.

In [ ]:
import pandas as pd
import numpy as np
import json

df = pd.read_parquet(r"C:\Users\Admin\Downloads\archive\out\btc_1min_clean.parquet")
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.set_index('Timestamp').sort_index()


In [ ]:
# ---- Daily aggregation ----
d = df.resample('D').agg(Open=('Open','first'), High=('High','max'), Low=('Low','min'),
                         Close=('Close','last'), Volume=('Volume','sum'),
                         MinCount=('Volume','count'))
d = d[d.index < '2026-09-22']
td = len(d)
d['ret'] = d['Close'].pct_change()
d['logret'] = np.log(d['Close']).diff()
d['prev_close'] = d['Close'].shift(1)
d['range'] = d['High'] - d['Low']
d['range_pct'] = d['range'] / d['prev_close']
d['body'] = (d['Close'] - d['Open']).abs() / d['prev_close']
d['vol_price'] = d['Volume'] / d['prev_close']
d['prev_vol'] = d['Volume'].shift(1)
d['vol_change'] = d['Volume'] / d['prev_vol'].replace(0, np.nan)
d['up'] = (d['Close'] > d['Open']).astype(int)

# rolling features
for w in [7, 30]:
    d[f'vol_sma{w}'] = d['Volume'].rolling(w).mean()
    d[f'close_sma{w}'] = d['Close'].rolling(w).mean()
    d[f'vol_{w}'] = d['ret'].rolling(w).std() * np.sqrt(365)
    d[f'ret_{w}'] = d['Close'].pct_change(w)
d['gap'] = (d['Open'] / d['prev_close'] - 1)
d['maxdrawdown_hist'] = (d['Close'] / d['Close'].cummax() - 1)

# week / month features
d['dow'] = d.index.dayofweek
d['month'] = d.index.month
d['year'] = d.index.year

d.to_parquet(r"C:\Users\Admin\Downloads\archive\out\btc_daily.parquet")


In [ ]:
# ---- Monthly & yearly ----
m = d.resample('ME').agg(Close=('Close','last'), Volume=('Volume','sum'),
                         High=('High','max'), Low=('Low','min'), Open=('Open','first'))
m['ret'] = m['Close'].pct_change()
m['logret'] = np.log(m['Close']).diff()
m.to_parquet(r"C:\Users\Admin\Downloads\archive\out\btc_monthly.parquet")

y = d.resample('YE').agg(Close=('Close','last'), Volume=('Volume','sum'),
                         High=('High','max'), Low=('Low','min'), First=('Open','first'))
y['full_year_ret'] = (y['Close'] / y['First'] - 1) * 100
y['pct_volume'] = 100 * y['Volume'] / y['Volume'].sum()
y.to_parquet(r"C:\Users\Admin\Downloads\archive\out\btc_yearly.parquet")


In [ ]:
# ---- Summary ----
summary = {
 'daily_rows': int(td),
 'date_range': [str(d.index.min().date()), str(d.index.max().date())],
 'min_close': float(d['Close'].min()),
 'max_close': float(d['Close'].max()),
 'mean_daily_vol_btc': float(d['Volume'].mean()),
 'median_daily_vol_btc': float(d['Volume'].median()),
 'total_volume_btc': float(d['Volume'].sum()),
 'mean_daily_ret_pct': float(d['ret'].mean()*100),
 'std_daily_ret_pct': float(d['ret'].std()*100),
 'annualized_vol_pct': float(d['ret'].std()*np.sqrt(365)*100),
 'days_up': int((d['ret']>0).sum()),
 'days_down': int((d['ret']<0).sum()),
 'days_flat': int((d['ret']==0).sum()),
 'up_ratio': float((d['ret']>0).mean()),
 'best_day_pct': float(d['ret'].max()*100),
 'worst_day_pct': float(d['ret'].min()*100),
 'days_with_zero_vol_daily': int((d['Volume']==0).sum()),
 'max_drawdown_pct': float(d['maxdrawdown_hist'].min()*100),
 'cagr_pct': float((d['Close'].iloc[-1]/d['Close'].iloc[0])**(365.25/td)*100-100),
 'total_growth_x': float(d['Close'].iloc[-1]/d['Close'].iloc[0]),
 'avg_days_to_double_halving': float(np.median(np.diff(d.index)[d['ret']>0][:0].astype(float))) if False else None,
}
# annual vol by year
d['mret'] = d['ret']
av = (d.groupby(d.index.year)['ret'].agg(lambda s: s.std(ddof=1) * np.sqrt(365) * 100)).round(1)
summary['annual_vol_by_year_pct'] = {str(i): float(v) for i, v in av.items()}
yr = d.groupby(d.index.year)['Volume'].sum() / 1e6
summary['volume_by_year_mbtc'] = {str(i): round(float(v),2) for i, v in yr.items()}
# longest up/down streaks
def streaks(r):
    best_up = cur_up = 0; best_dn = cur_dn = 0
    for x in r:
        if x > 0:
            cur_up += 1; cur_dn = 0
        elif x < 0:
            cur_dn += 1; cur_up = 0
        best_up = max(best_up, cur_up); best_dn = max(best_dn, cur_dn)
    return best_up, best_dn
summary['longest_up_streak_days'], summary['longest_down_streak_days'] = streaks(d['ret'].tolist())

with open(r"C:\Users\Admin\Downloads\archive\out\summary_daily.json","w") as f:
    json.dump(summary, f, indent=2, default=str)

print(json.dumps(summary, indent=2, default=str))
print("\nannual_vol:", av.to_dict())
print("\nvolume by yr MBTC:", yr.round(2).to_dict())
print("\nmonthly rows:", len(m))
print("\nquick peak of daily head:\n", d.head(3))

---

## 03 - Exploratory Data Analysis

Univariate/bivariate/multivariate EDA: distributions, weekday & intraday hourly patterns, volatility, drawdown, correlations, calendar heatmap, ACF. Produces c1-c10 charts.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox':'tight', 'axes.grid':True,
                     'grid.alpha':0.3, 'axes.spines.top':False, 'axes.spines.right':False,
                     'font.size':9})
OUT = r"C:\Users\Admin\Downloads\archive\out"
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index)
mi = pd.read_parquet(f"{OUT}\\btc_1min_clean.parquet")
mi['Timestamp'] = pd.to_datetime(mi['Timestamp'])
mi = mi[(mi['Timestamp'] >= '2017-01-01')].set_index('Timestamp')


In [ ]:
# ---------- 1. Log close with cycle shading ----------
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(d.index, d['Close'], lw=0.8, color='#1f77b4')
halvings = [('2012-11-28','H1'),('2016-07-09','H2'),('2020-05-11','H3'),('2024-04-19','H4')]
for dt, lab in halvings:
    ax.axvline(pd.Timestamp(dt), color='red', ls='--', lw=1, alpha=0.7)
    ax.text(pd.Timestamp(dt), 2, lab, color='red', fontsize=8, ha='left')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_title('BTC/USD Daily Close (log scale) with Bitcoin Halvings (H1–H4)')
ax.set_xlabel(''); ax.set_ylabel('Close price (USD, log)')
ax.xaxis.set_major_locator(mdates.YearLocator())
fig.savefig(f"{OUT}\\c1_log_close.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 2. Daily returns distribution ----------
fig, ax = plt.subplots(1,2, figsize=(12,4), gridspec_kw={'width_ratios':[2,1]})
r = d['ret'].dropna()
ax[0].hist(r, bins=140, density=True, color='#2ca02c', alpha=0.7)
xx = np.linspace(r.min(), r.max(), 500)
m, s = r.mean(), r.std()
ax[0].plot(xx, (1/(s*np.sqrt(2*np.pi)))*np.exp(-0.5*((xx-m)/s)**2), 'r-', lw=1.5, label='Normal fit')
ax[0].set_title(f"Daily returns distribution (n={len(r)})\nmean={m*100:.2f}%, std={s*100:.2f}%, skew={r.skew():.2f}, kurt={r.kurtosis():.2f}")
ax[0].set_xlabel('Daily return'); ax[0].legend()
names, vals = ['±1σ','±2σ','±3σ'], [0.0,0.0,0.0]
for i,k in enumerate([1,2,3]):
    vals[i]=((r.abs() > k*s).mean()*100)
vals2 = [r.le(k*s).ge(-k*s).mean() for k in [1,2,3]]
ax[1].bar(['±1σ','±2σ','±3σ'], vals, color=['#d62728','#ff7f0e','#bcbd22'])
ax[1].set_title('% of days beyond normal σ bands\n(fat tails)')
for i,v in enumerate(vals): ax[1].text(i, v, f'{v:.2f}%', ha='center', va='bottom', fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}\\c2_returns_dist.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 3. Annualized vol by year + rolling 30d vol ----------
fig, ax = plt.subplots(2,1, figsize=(12,7))
av = (d.groupby(d.index.year)['ret'].std()*np.sqrt(365)*100)
ax[0].bar(av.index.astype(str), av.values, color='#ff7f0e')
for i,v in enumerate(av.values): ax[0].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=7)
ax[0].set_title('Annualized volatility by year (%)'); ax[0].set_ylabel('Ann. vol %')
rv = d['ret'].rolling(90).std()*np.sqrt(365)*100
ax[1].plot(d.index, rv, lw=0.9, color='#9467bd')
ax[1].set_title('90-day rolling annualized volatility (%) — volatility clustering / regimes')
ax[1].set_ylabel('Rolling ann. vol %'); ax[1].xaxis.set_major_locator(mdates.YearLocator())
fig.tight_layout(); fig.savefig(f"{OUT}\\c3_volatility.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 4. Drawdown & volume ----------
fig, ax = plt.subplots(2,1, figsize=(12,7), sharex=True)
ax[0].fill_between(d.index, d['maxdrawdown_hist']*100, 0, color='#d62728', alpha=0.6)
ax[0].set_title('Drawdown from all-time-high (%)'); ax[0].set_ylabel('Drawdown %')
ax[1].plot(d.index, d['Volume'], lw=0.7, color='#8c564b')
ax[1].set_yscale('log'); ax[1].set_title('Daily traded volume (BTC, log)')
ax[1].set_ylabel('Volume (BTC)'); ax[1].xaxis.set_major_locator(mdates.YearLocator())
fig.tight_layout(); fig.savefig(f"{OUT}\\c4_drawdown_vol.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 5. Day-of-week effect ----------
fig, ax = plt.subplots(1,2, figsize=(11,4))
dow = d.groupby('dow')['ret'].agg(['mean','median'])
win = d.groupby('dow')['up'].mean()*100
dow_names=['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
ax[0].bar(dow_names, dow['mean']*100, color='#1f77b4')
for i,v in enumerate(dow['mean']*100): ax[0].text(i, v+(0.002), f'{v:.2f}', ha='center', fontsize=7)
ax[0].axhline(0, color='k', lw=0.8); ax[0].set_title('Mean daily return by weekday (%)')
ax[1].bar(dow_names, win, color='#2ca02c')
for i,v in enumerate(win): ax[1].text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=7)
ax[1].set_title('Win rate (% up days) by weekday')
fig.tight_layout(); fig.savefig(f"{OUT}\\c5_dow.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 6. Intraday hour-of-day pattern (1-min, 2017+) ----------
mi['hr'] = mi.index.hour
hh = mi['hr']
lret = np.log(mi['Close']).diff()
ig = pd.DataFrame({'hr': hh, 'lret': lret, 'abs': lret.abs()}).dropna()
by_hr = ig.groupby('hr').agg(mean_ret=('lret','mean'), mean_abs=('abs','mean'), n=('lret','count'))
by_hr['mean_ret_bps'] = by_hr['mean_ret']*10000
by_hr['mean_abs_bps'] = by_hr['mean_abs']*10000
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].bar(by_hr.index, by_hr['mean_ret_bps'], color='#1f77b4')
ax[0].axhline(0, color='k', lw=0.8)
ax[0].set_title('Mean 1-min return by hour-of-day UTC (bps, 2017+)')
ax[0].set_xlabel('Hour (UTC)')
ax[1].bar(by_hr.index, by_hr['mean_abs_bps'], color='#ff7f0e')
ax[1].set_title('Mean |1-min return| by hour — intraday volatility (bps, 2017+)')
ax[1].set_xlabel('Hour (UTC)')
by_hr.round(3).to_csv(f"{OUT}\\hourly_pattern.csv")
fig.tight_layout(); fig.savefig(f"{OUT}\\c6_hour.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 7. Log price vs log volume (daily) ----------
fig, ax = plt.subplots(figsize=(6,4.5))
lg = d.dropna(subset=['ret']).iloc[1:]
ax.scatter(np.log(lg['Close']), np.log(lg['Volume']+1), s=3, alpha=0.4, color='#9467bd')
z = np.polyfit(np.log(lg['Close']), np.log(lg['Volume']+1), 1)
xx = np.sort(np.log(lg['Close']))
ax.plot(xx, np.polyval(z, xx), 'r-', lw=2, label=f"slope={z[0]:.2f}")
ax.set_title('Log Volume vs Log Close (daily)')
ax.set_xlabel('log(Close)'); ax.set_ylabel('log(Volume+1)'); ax.legend()
sp = lg[['Close','Volume']].corr(method='spearman')
ax.text(0.05,0.9, f"Spearman ρ(volume, close) = {sp.loc['Close','Volume']:.3f}", transform=ax.transAxes)
fig.savefig(f"{OUT}\\c7_vol_close.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 8. Calendar heatmap (mean monthly return by year-month) ----------
d['ym'] = d.index.to_period('M')
mtr = (d.groupby('ym')['ret'].mean()*100).reset_index()
mtr['year'] = mtr['ym'].dt.year; mtr['month'] = mtr['ym'].dt.month
piv = mtr.pivot(index='year', columns='month', values='ret')
fig, ax = plt.subplots(figsize=(12,6))
mx = float(np.nanmax(np.abs(piv.values.astype(float))))
im = ax.imshow(piv.values, aspect='auto', cmap='RdYlGn', vmin=-mx, vmax=mx)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels([str(x) for x in piv.index])
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45, ha='right')
ax.set_title('Mean daily return by month-year (%) — green = up months')
fig.colorbar(im, label='Mean daily return %', shrink=0.7)
fig.tight_layout(); fig.savefig(f"{OUT}\\c8_calendar.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 9. QQ plot vs normal ----------
from scipy import stats as st
fig, ax = plt.subplots(figsize=(5,5))
st.probplot(r.sample(3000, random_state=1), dist='norm', plot=ax)
ax.set_title('QQ plot — daily returns vs Normal (heavy tails)')
fig.tight_layout(); fig.savefig(f"{OUT}\\c9_qq.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- 10. ACF: returns vs squared returns ----------
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(2,1, figsize=(11,6))
plot_acf(r, lags=30, ax=ax[0], title='ACF daily returns (≈0 → no serial correlation in levels)')
plot_acf(r**2, lags=30, ax=ax[1], title='ACF squared returns (sign of volatility clustering)')
fig.tight_layout(); fig.savefig(f"{OUT}\\c10_acf.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- Extra stats ----------
extra = {}
w = d.copy()
extra['by_year'] = {}
for yr, g in w.groupby(w.index.year):
    extra['by_year'][str(yr)] = {
        'ret_pct': float(g['ret'].sum()*100),
        'win_rate': round(float((g['ret']>0).mean()*100),1),
        'ann_vol': round(float(g['ret'].std()*np.sqrt(365)*100),1),
        'close_start': float(g['Close'].iloc[0]),
        'close_end': float(g['Close'].iloc[-1]),
        'max_drawdown': round(float((g['Close']/g['Close'].cummax()-1).min()*100),1),
    }
top = w.reindex(w['ret'].sort_values(ascending=False).index)[['Close','ret','Volume','year','month']].head(10)
bottom = w.reindex(w['ret'].sort_values().index)[['Close','ret','Volume','year','month']].head(10)
extra['top10_days'] = [[str(idx.date()), round(float(x['Close']),0), round(float(x['ret']*100),2), round(float(x['Volume']),0)] for idx,x in top.iterrows()]
extra['bottom10_days'] = [[str(idx.date()), round(float(x['Close']),0), round(float(x['ret']*100),2), round(float(x['Volume']),0)] for idx,x in bottom.iterrows()]
# market cap approx? skip (need supply)
# drawdowns episodes
dd = w['maxdrawdown_hist']
valleys = []
cur = dd.iloc[0]; start = w.index[0]
for i in range(1,len(dd)):
    if dd.iloc[i] < cur:
        cur = dd.iloc[i]
    elif dd.iloc[i] == 0:
        if cur < -0.3:
            valleys.append((start, w.index[i-1], float(cur*100)))
        cur = dd.iloc[i]
        start = w.index[i]
if cur < -0.3: valleys.append((start, w.index[-1], float(cur*100)))
extra['drawdown_episodes'] = [[str(a.date()), str(b.date()), round(v,1)] for a,b,v in valleys]
# returns around halving (x months)
extra['halving_cycles'] = {}
halv = ['2012-11-28','2016-07-09','2020-05-11','2024-04-19']
for h in halv:
    t = pd.Timestamp(h)
    pre = w.loc[(w.index < t) & (w.index >= t - pd.DateOffset(months=12)), 'Close']
    post = w.loc[(w.index >= t) & (w.index < t + pd.DateOffset(months=18)), 'Close']
    extra['halving_cycles'][h] = {
      'pre12m_ret': round(float((pre.iloc[-1]/pre.iloc[0]-1)*100),1) if len(pre)>2 else None,
      'post18m_ret': round(float((post.iloc[-1]/post.iloc[0]-1)*100),1) if len(post)>2 else None,
    }
extra['dow_table'] = {dow_names[int(k)]: {'mean_ret': round(float(v['mean']*100),3),
                                          'median': round(float(v['median']*100),3),
                                          'win_rate': round(float(win[int(k)]),1)} for k,v in dow.iterrows()}
extra['acf_ret_lag1'] = float(r.autocorr(1))
extra['acf_ret2_lag1'] = float((r**2).autocorr(1))
extra['box_m_test_cohort'] = None
with open(f"{OUT}\\extra_eda.json","w") as f: json.dump(extra, f, indent=2, default=str)

print("=== by year ===")
print(pd.DataFrame(extra['by_year']).T.to_string())
print("\n=== drawdown episodes (30%+ troughs) ===")
for a,b,v in extra['drawdown_episodes']:
    print(f"{a} -> {b}: {v}%")
print("\n=== top days ==="); import pprint; pprint.pprint(extra['top10_days'])
print("\n=== worst days ==="); pprint.pprint(extra['bottom10_days'])
print("\n=== halving cycles ==="); pprint.pprint(extra['halving_cycles'])
print("\n=== DOW table ==="); pprint.pprint(extra['dow_table'])
print("\nsaved charts")

---

## 04 - Statistical Analysis & GARCH

Hypothesis tests (t, Wilcoxon, Kruskal-Wallis, Mann-Whitney), stationarity (ADF/KPSS), Ljung-Box, halving effect sizes and a GARCH(1,1) volatility model. Outputs stats_04.json.

In [ ]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings('ignore')
from scipy import stats as st
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

OUT = r"C:\Users\Admin\Downloads\archive\out"
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index)
r = d['ret'].dropna()
lr = d['logret'].dropna()
res = {}

PO = lambda a,b,c: print(f"{a:34s}  {b}")


In [ ]:
# ---- 1. Distribution stats ----
po = []
for name, s in [('Daily simple ret', r), ('Daily log ret', lr)]:
    res[f'dist_{name}'] = dict(mean=float(s.mean()), median=float(s.median()),
        std=float(s.std(ddof=1)), skew=float(st.skew(s)), kurt=float(st.kurtosis(s, fisher=False)),
        min=float(s.min()), max=float(s.max()),
        n=len(s))
    jb = st.jarque_bera(s)
    res[f'jb_{name}'] = {'stat': float(jb[0]), 'p': float(jb[1])}
    po += [f"{name}: mean={s.mean():.4%} median={s.median():.4%} std={s.std():.4%} skew={st.skew(s):.2f} kurt(fisher0)={st.kurtosis(s, fisher=False):.1f} JB-p={jb[1]:.2e}"]


In [ ]:
# ---- 2. Mean != 0? ----
t = st.ttest_1samp(r, 0)
res['ttest_mean0'] = {'stat': float(t.statistic), 'p': float(t.pvalue), 'ci95': [float(a) for a in st.t.interval(0.95, len(r)-1, loc=r.mean(), scale=st.sem(r))]}
wilcox = st.wilcoxon(r[np.abs(r)>1e-12], alternative='two-sided')
res['wilcoxon_median0'] = {'stat': float(wilcox.statistic), 'p': float(wilcox.pvalue)}
ci = st.t.interval(0.95, len(r)-1, loc=r.mean(), scale=st.sem(r))
res['dist_daily_ret_ci95'] = [float(a) for a in ci]
po += [f"t-test mean=0: t={t.statistic:.2f} p={t.pvalue:.2e} 95%CI daily return = [{ci[0]:.5f}, {ci[1]:.5f}]"]
po += [f"Wilcoxon median=0: p={wilcox.pvalue:.2e}"]


In [ ]:
# ---- 3. Stationarity ----
for nm, s in [('close_levels', d['Close']), ('logprice', np.log(d['Close'])), ('logret', lr)]:
    a = adfuller(s.dropna())
    res[f'adf_{nm}'] = {'stat': float(a[0]), 'p': float(a[1])}
    po += [f"ADF {nm}: stat={a[0]:8.2f} p={a[1]:.2e} -> {'stationary' if a[1]<0.05 else 'non-stationary'}"]
k = kpss(lr.dropna(), 'ct') if False else kpss(lr.dropna(), 'c')
res['kpss_logret'] = {'stat': float(k[0]), 'p': float(k[1])}
po += [f"KPSS(logret, c): stat={k[0]:.3f} p={k[1]:.2f} -> {'stationary' if k[1]>0.05 else 'non-stationary'}"]


In [ ]:
# ---- 4. Ljung-Box white noise (returns) & vol clustering (sq returns) ----
lb_r = acorr_ljungbox(r, lags=[10,30], return_df=True)
lb_r2 = acorr_ljungbox(r**2, lags=[10,30], return_df=True)
res['ljungbox_ret'] = {str(i): {'stat': float(lb_r.loc[i,'lb_stat']), 'p': float(lb_r.loc[i,'lb_pvalue'])} for i in [10,30]}
res['ljungbox_sq_ret'] = {str(i): {'stat': float(lb_r2.loc[i,'lb_stat']), 'p': float(lb_r2.loc[i,'lb_pvalue'])} for i in [10,30]}
po += [f"Ljung-Box ret lag30: stat={lb_r.loc[30,'lb_stat']:.1f} p={lb_r.loc[30,'lb_pvalue']:.2e} (white noise?)"]
po += [f"Ljung-Box ret^2 lag30: stat={lb_r2.loc[30,'lb_stat']:.1f} p={lb_r2.loc[30,'lb_pvalue']:.2e} (vol clustering!)"]
ac1 = float(np.corrcoef(r[:-1], r[1:])[0,1]); ac1sq = float(np.corrcoef(r[:-1]**2, r[1:]**2)[0,1])
res['autocorr_ret_lag1'], res['autocorr_sqret_lag1'] = ac1, ac1sq
po += [f"ACF(1) returns={ac1:.3f}, squared returns={ac1sq:.3f}"]


In [ ]:
# ---- 5. Weekday effects (Kruskal-Wallis, non-parametric) ----
d2 = d.dropna(subset=['ret'])
kw_ret = st.kruskal(*[g['ret'].values for _, g in d2.groupby('dow')])
kw_abs = st.kruskal(*[g['ret'].abs().values for _, g in d2.groupby('dow')])
res['kruskal_dow_ret'] = {'stat': float(kw_ret.statistic), 'p': float(kw_ret.pvalue)}
res['kruskal_dow_absret'] = {'stat': float(kw_abs.statistic), 'p': float(kw_abs.pvalue)}
po += [f"Kruskal-Wallis DOW on return: H={kw_ret.statistic:.2f} p={kw_ret.pvalue:.2f}"]
po += [f"Kruskal-Wallis DOW on |return| (vol): H={kw_abs.statistic:.2f} p={kw_abs.pvalue:.2f}"]

# monthly seasonality
kw_mo = st.kruskal(*[g['ret'].values for _, g in d2.groupby('month')])
res['kruskal_month_ret'] = {'stat': float(kw_mo.statistic), 'p': float(kw_mo.pvalue)}
po += [f"Kruskal-Wallis month on return: H={kw_mo.statistic:.2f} p={kw_mo.pvalue:.2f}"]


In [ ]:
# ---- 6. Hour-of-day on 1-min (2017+) ----
mi = pd.read_parquet(f"{OUT}\\btc_1min_clean.parquet")
mi['Timestamp'] = pd.to_datetime(mi['Timestamp'])
mi = mi[mi['Timestamp'] >= '2017-01-01'].set_index('Timestamp')
mi['hr'] = mi.index.hour
mi = mi.assign(lr=np.log(mi['Close']).diff()).dropna()
kw_hr = st.kruskal(*[g['lr'].values for _, g in mi.groupby('hr')])
kw_hr_abs = st.kruskal(*[g['lr'].abs().values for _, g in mi.groupby('hr')])
res['kruskal_hour_ret'] = {'stat': float(kw_hr.statistic), 'p': float(kw_hr.pvalue), 'n': int(len(mi))}
res['kruskal_hour_absret'] = {'stat': float(kw_hr_abs.statistic), 'p': float(kw_hr_abs.pvalue)}
po += [f"Kruskal-Wallis hour on 1-min ret: H={kw_hr.statistic:.2f} p={kw_hr.pvalue:.2e} (n={len(mi):,})"]
po += [f"Kruskal-Wallis hour on |ret|: H={kw_hr_abs.statistic:.2f} p={kw_hr_abs.pvalue:.2e}"]


In [ ]:
# ---- 7. Return-Volume correlation ----
v = d2[['ret','Volume','Close']].dropna()
sp_rho, sp_p = st.spearmanr(v['ret'], v['Volume'])
pear_r, pear_p = st.pearsonr(v['ret'], v['Volume'])
res['corr_ret_vol'] = {'spearman_rho': float(sp_rho), 'spearman_p': float(sp_p),
                       'pearson_r': float(pear_r), 'pearson_p': float(pear_p)}
logp, logv = np.log(v['Close']), np.log(v['Volume']+1)
sp2, sp2p = st.spearmanr(logp, logv)
res['corr_logclose_logvol'] = {'spearman': float(sp2), 'p': float(sp2p)}
po += [f"Spearman ret vs Volume: rho={sp_rho:.4f} p={sp_p:.2e}"]
po += [f"Spearman log(close) vs log(vol): rho={sp2:.3f} p={sp2p:.2e}"]

# partial: correlation of |ret| and volume (vol-up-volume relationship)
rho3, p3 = st.spearmanr(v['ret'].abs(), v['Volume'])
res['corr_absret_vol'] = {'spearman': float(rho3), 'p': float(p3)}
po += [f"Spearman |ret| vs Volume: rho={rho3:.4f} p={p3:.2e} (volume-activity link)"]


In [ ]:
# ---- 8. Halving regime test: mean return post vs pre halving ----
halv = ['2012-11-28','2016-07-09','2020-05-11','2024-04-19']
pre = [d2[(d2.index < pd.Timestamp(h)) & (d2.index >= pd.Timestamp(h)-pd.DateOffset(months=18))]['ret'] for h in halv]
post = [d2[(d2.index >= pd.Timestamp(h)) & (d2.index < pd.Timestamp(h)+pd.DateOffset(months=18))]['ret'] for h in halv]
prec = pd.concat(pre); postc = pd.concat(post)
mann = st.mannwhitneyu(prec, postc, alternative='two-sided')
pooled = np.sqrt(((len(prec)-1)*prec.std()**2 + (len(postc)-1)*postc.std()**2)/(len(prec)+len(postc)-2))
cohend = (postc.mean()-prec.mean())/pooled
res['halving_pre_post'] = {'pre_mean': float(prec.mean()), 'post_mean': float(postc.mean()),
    'pre_std': float(prec.std()), 'post_std': float(postc.std()),
    'mann_p': float(mann.pvalue), 'cohen_d': float(cohend), 'pre_n': int(len(prec)), 'post_n': int(len(postc))}
po += [f"Halving: pre(18m) mean daily ret={prec.mean():.4%} (n={len(prec)}), post(18m)={postc.mean():.4%} (n={len(postc)}) Mann-Whitney p={mann.pvalue:.3f} Cohen d={cohend:.2f}"]


In [ ]:
# ---- 9. OLS: log price vs time ----
import statsmodels.api as sm
X = np.arange(len(d)); y = np.log(d['Close'].values)
Xc = sm.add_constant(X)
m1 = sm.OLS(y, Xc).fit()
res['ols_logprice_time'] = {'slope': float(m1.params[1]), 't': float(m1.tvalues[1]), 'p': float(m1.pvalues[1]), 'r2': float(m1.rsquared)}
po += [f"OLS log(close)~time: slope={m1.params[1]:.4f}/day (~{np.expm1(m1.params[1]*365)*100:.0f}%/yr trend component) R2={m1.rsquared:.3f} t={m1.tvalues[1]:.0f}"]


In [ ]:
# ---- 10. Relationship ret_{t+1} ~ ret_t momentum ----
v2 = d2.copy()
v2['ret_next'] = v2['ret'].shift(-1)
momentum = v2[['ret','ret_next']].dropna()
spm, spmp = st.spearmanr(momentum['ret'], momentum['ret_next'])
res['corr_ret_lag1_next'] = {'spearman': float(spm), 'p': float(spmp)}
po += [f"Spearman ret_t vs ret_{t+1 if False else 'next'}: rho={spm:.4f} p={spmp:.2e} (no momentum)"]

# composite chart: QQ, ACF already; add variance ratio?; print all
print("\n".join(po))
with open(f"{OUT}\\stats_04.json","w") as f: json.dump(res, f, indent=2, default=str)


In [ ]:
# ---- GARCH(1,1) if available ----
try:
    from arch import arch_model
    am = arch_model(lr*100, mean='Constant', vol='GARCH', p=1, q=1, rescale=False)
    m = am.fit(disp='off')
    res['garch11'] = {'omega': float(m.params.get('omega')), 'alpha1': float(m.params.get('alpha[1]') or m.params.get('alpha1')),
                      'beta1': float(m.params.get('beta[1]') or m.params.get('beta1')),
                      'mu': float(m.params.get('mu')), 'loglik': float(m.loglikelihood),
                      'half_life_days': None}
    if res['garch11']['beta1'] is not None and res['garch11']['alpha1'] is not None:
        ht = np.log(0.5)/np.log(res['garch11']['beta1'] + res['garch11']['alpha1'])
        res['garch11']['half_life_days'] = float(ht)
    print(f"\nGARCH(1,1): omega={m.params.get('omega'):.4f} alpha={m.params.get('alpha[1]') or m.params.get('alpha1'):.3f} beta={m.params.get('beta[1]') or m.params.get('beta1'):.3f} persistence={res['garch11']['beta1']+res['garch11']['alpha1']:.3f}")
    with open(f"{OUT}\\stats_04.json","w") as f: json.dump(res, f, indent=2, default=str)
except Exception as e:
    print("GARCH skipped:", e)

---

## 05 - Anomaly Detection & Regime Segmentation

Flags extreme days via robust modified z-score and Isolation Forest; clusters daily regimes with KMeans and profiles them. Produces c11-c12.

In [ ]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings('ignore')
from scipy import stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

OUT = r"C:\Users\Admin\Downloads\archive\out"
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index)
r = d['ret'].dropna()
res = {}
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox':'tight', 'axes.grid':True, 'grid.alpha':0.3,
                     'axes.spines.top':False, 'axes.spines.right':False, 'font.size':9})


In [ ]:
# ================= ANOMALY DETECTION =================
# 1) Robust modified z-score on daily log returns (MAD-based)
lnr = d['logret'].dropna()
med = lnr.median(); mad = (lnr - med).abs().median()
if mad == 0: mad = lnr.std()
mz = 0.6745 * (lnr - med) / mad
anom_z = lnr[mz.abs() > 5]  # borderline/clear anomalies per Iglewicz-Banerjee
res['anomaly_mz'] = {
    'mad': float(mad), 'threshold': '|MZ|>5',
    'n_anomalies': int(len(anom_z)),
    'table': [[str(ix.date()), round(float(v*100),2), round(float(mz.loc[ix]),1)] for ix, v in anom_z.items()]
}

# 2) Isolation Forest on multi-feature daily snapshot
feats = d[['ret','range_pct','vol_change','logret','Volume']].replace([np.inf,-np.inf], np.nan).dropna()
X = feats.copy()
X['log_vol'] = np.log1p(X['Volume']); X['abs_ret'] = X['ret'].abs()
X = X[['log_vol','abs_ret','range_pct','vol_change','ret']]
Xs = pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)
iso = IsolationForest(n_estimators=400, contamination=0.01, random_state=7)
labs = iso.fit_predict(Xs)
res['anomaly_if'] = {
    'contamination': 0.01,
    'n_flagged': int((labs==-1).sum()),
    'last_30_flagged': [[str(ix.date()), round(float(X.loc[ix,'ret']*100),2), round(float(X.loc[ix,'log_vol']),2)] for ix in X.index[labs==-1][-30:]]
}
X['iso_flag'] = labs

# 3) overlap between methods
a_mz_idx = set(anom_z.index)
a_if_idx = set(X.index[labs==-1])
res['anomaly_overlap'] = {'mz_n': len(a_mz_idx), 'if_n': len(a_if_idx), 'both_n': int(len(a_mz_idx & a_if_idx))}

# chart: labelled anomalies on log price
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(d.index, d['Close'], lw=0.8, color='#1f77b4')
bad = d.loc[sorted(a_mz_idx | a_if_idx)]
ax.scatter(bad.index, bad['Close'], s=22, color='#d62728', zorder=5, label=f'anomalies (n={len(bad)})')
ax.set_yscale('log'); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"${v:,.0f}"))
ax.set_title('Daily price (log) with statistical anomalies — extreme-return days (red)')
ax.legend(); ax.xaxis.set_major_locator(mdates.YearLocator())
fig.savefig(f"{OUT}\\c11_anomalies.png"); display(fig); plt.close(fig)


In [ ]:
# ================= REGIME SEGMENTATION =================
R = X.join(d['ret'], rsuffix='_b').copy()
seg_vars = R[['log_vol','abs_ret','range_pct','ret']].dropna()
segv = pd.DataFrame(StandardScaler().fit_transform(seg_vars), index=seg_vars.index, columns=seg_vars.columns)
k = KMeans(n_clusters=4, n_init=20, random_state=1)
R['regime'] = k.fit_predict(segv)
# order regimes by mean return
order = R.groupby('regime')['ret'].mean().sort_values(ascending=False).index
map_ = {old: new for new, old in enumerate(order)}
R['regime'] = R['regime'].map(map_)
names = ['Euphoria (extreme up)','Bull (steady up)','Base (calm/mixed)','Stress (extreme down)']
profile = R.groupby('regime').agg(n=('ret','count'), mean_ret=('ret','mean'), med_ret=('ret','median'),
    ann_vol=('log_vol', lambda s: np.expm1(s.mean())**2*0), std_ret=('ret','std'),
    mean_abs_ret=('abs_ret','mean'), mean_logvol=('log_vol','mean'), vol_change=('vol_change','mean'))
profile['share_pct'] = 100*profile['n']/profile['n'].sum()
profile.index = names[:len(profile)]
# next day outcome by regime (transition matrix)
R2 = R.join(d['ret'].shift(-1).rename('ret_next'))
tmat = R2.groupby('regime')['ret_next'].agg(mean='mean', std='std').round(4)
# fmt
prof_out = {}
for nm, row in profile.iterrows():
    trow = tmat.iloc[list(profile.index).index(nm)]
    prof_out[nm] = {'n_days': int(row['n']), 'share_pct': round(float(row['share_pct']),1),
        'mean_ret': round(float(row['mean_ret']*100),3), 'med_ret': round(float(row['med_ret']*100),3),
        'ann_vol_est': round(float(row['std_ret']*np.sqrt(365)*100),1),
        'mean_abs_ret': round(float(row['mean_abs_ret']*100),3),
        'mean_log_vol': round(float(row['mean_logvol']),2),
        'mean_vol_change': round(float(row['vol_change']),2),
        'next_day_mean_ret': round(float(trow['mean']*100),3),
        'next_day_std': round(float(trow['std']*100),2)}
res['regimes'] = prof_out
print("=== Regime profiles ===")
for kk, vv in prof_out.items(): print(f"  {kk}: {vv}")
# regime frequencies by year
R['year'] = R.index.year
freq_by_year = pd.crosstab(R['year'], R['regime'], normalize='index')*100
res['regime_freq_by_year'] = {str(y): {names[int(c)]: round(float(v),1) for c, v in row.items()} for y, row in freq_by_year.iterrows()}

fig, ax = plt.subplots(2,1, figsize=(13,7))
freq_by_year.plot(kind='bar', stacked=True, ax=ax[0], colormap='RdYlGn', width=0.9)
ax[0].set_title('Share of trading days in each regime by year')
ax[0].set_ylabel('% of days'); ax[0].legend(title='Regime', labels=[names[int(c)] for c in freq_by_year.columns])
ax[1].plot(R.index, R['regime'], 'o', ms=2, alpha=0.5)
ax[1].set_yticks(range(len(names))); ax[1].set_yticklabels(names)
ax[1].set_title('Daily regime label over time'); ax[1].xaxis.set_major_locator(mdates.YearLocator())
fig.tight_layout(); fig.savefig(f"{OUT}\\c12_regimes.png"); display(fig); plt.close(fig)

with open(f"{OUT}\\anomaly_seg_05.json","w") as f: json.dump(res, f, indent=2, default=str)
R.to_parquet(f"{OUT}\\daily_regimes.parquet")
print("\nAnomalies (MZ>5) count:", len(anom_z), "| IF flagged:", (labs==-1).sum())

---

## 06 - Walk-Forward Machine Learning

Predicts next-day direction with LR/RandomForest/HistGBM under expanding-window walk-forward validation (1-day embargo). Outputs ml_06.json, ml_final_06.json, c13-c14.

In [ ]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, log_loss, balanced_accuracy_score,
                             average_precision_score, confusion_matrix, mean_squared_error, mean_absolute_error)
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

OUT = r"C:\Users\Admin\Downloads\archive\out"
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index)


In [ ]:
# ---------- leakage-safe feature engineering (only info known at close of day t) ----------
f = pd.DataFrame(index=d.index)
f['ret_1'] = d['ret']
for k in [2,3,5,7,14,21,30]:
    f[f'ret_{k}'] = d['Close'].pct_change(k)
f['vol_7'] = d['ret'].rolling(7).std()*np.sqrt(365)
f['vol_30'] = d['ret'].rolling(30).std()*np.sqrt(365)
f['vol_90'] = d['ret'].rolling(90).std()*np.sqrt(365)
for w in [7,30,200]:
    sma = d['Close'].rolling(w).mean()
    f[f'sma{w}_ratio'] = d['Close']/sma - 1
f['log_vol'] = np.log1p(d['Volume'])
f['vol_change'] = (d['Volume']/d['prev_vol'].replace(0,np.nan) - 1)
f['range_pct'] = d['range_pct']
f['gap'] = d['gap']
# RSI 14
delta = d['Close'].diff()
up = delta.clip(lower=0); dn = -delta.clip(upper=0)
ru = up.rolling(14).mean(); rd = dn.rolling(14).mean()
f['rsi14'] = 100 - 100/(1 + ru/rd.replace(0,np.nan))
rs = ru/rd.replace(0,np.nan)
f['rsi14'] = 100 - 100/(1+rs)
# streak (days since last sign flip) — sign of last day plus count
sign = np.sign(d['ret']).fillna(0)
up_streak = sign.groupby((sign != sign.shift()).cumsum()).cumcount()+1
dn_streak = (-sign).groupby(((-sign) != (-sign).shift()).cumsum()).cumcount()+1
f['streak'] = np.where(sign>=0, up_streak, -dn_streak)
f['dow'] = d['dow']; f['month'] = d['month']
f['year'] = d.index.year

# target: next-day direction & return
f['ret_next'] = d['ret'].shift(-1)
f['y'] = (f['ret_next'] > 0).astype(int)
feat_cols = [c for c in f.columns if c not in ['ret_next','y','year']]

X = f[feat_cols + ['ret_next','y','year']].replace([np.inf,-np.inf], np.nan)
X = X[X.index >= '2013-01-01']
X = X.dropna(subset=feat_cols)


In [ ]:
# ---------- walk-forward expanding-window CV ----------
models = {
 'LogisticRegression': Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=3000, C=0.5))]),
 'RandomForest': RandomForestClassifier(n_estimators=400, min_samples_leaf=8, max_features=0.4, n_jobs=-1, random_state=3),
 'HistGradientBoost': HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=3),
}
folds = [f"{y}-01-01" for y in range(2021, 2027)]
df_out = {}
for name, mdl in models.items():
    preds, truths, feats_ = [], [], []
    fold_res = {}
    for fld in folds:
        tr = X[X.index < fld]
        te = X[(X.index >= fld) & (X.index < (pd.Timestamp(fld) + pd.DateOffset(years=1)))]
        if len(te) < 30 or len(tr) < 200: continue
        te = te.iloc[1:]  # embargo
        mdl.fit(tr[feat_cols], tr['y'])
        p = mdl.predict_proba(te[feat_cols])[:,1]
        preds += list(p); truths += list(te['y'].values); feats_ += list(te.index.date)
        fold_res[fld[:4]] = {'n': int(len(te)), 'auc': float(roc_auc_score(te['y'], p)),
                             'acc': float(accuracy_score(te['y'], (p>0.5).astype(int)))}
    df_out[name] = {
        'oob_auc': float(roc_auc_score(truths, preds)),
        'oob_acc': float(accuracy_score(truths, np.array(preds)>0.5)),
        'oob_balacc': float(balanced_accuracy_score(truths, np.array(preds)>0.5)),
        'oob_logloss': float(log_loss(truths, np.clip(preds,1e-6,1-1e-6))),
        'oob_pr_auc': float(average_precision_score(truths, preds)),
        'n_obs': len(truths),
        'folds': fold_res,
    }
    print(f"{name}: pooled OOS AUC={df_out[name]['oob_auc']:.4f} ACC={df_out[name]['oob_acc']:.4f} "
          f"PR-AUC={df_out[name]['oob_pr_auc']:.4f} {df_out[name]['folds']}")

# baseline (always predict majority: up)
up_share = float(X.loc[X.index<'2021-01-01','y'].mean())
truths_all = X[X.index>='2021-01-01'].iloc[1:]['y']
df_out['baseline_always_up'] = {'accuracy': float(truths_all.mean()), 'auc': 0.5,
    'note': 'naive - predicts up every day (evaluated on 2021+)'}
print("baseline always-up acc on 2021+:", round(truths_all.mean(),4))

with open(f"{OUT}\\ml_06.json","w") as fd: json.dump(df_out, fd, indent=2, default=str)


In [ ]:
# ---------- final model + feature importance on 2025-2026 test ----------
tr = X[X.index < '2025-01-01']
te = X[(X.index >= '2025-01-01')]
te = te.iloc[1:]
rf = RandomForestClassifier(n_estimators=600, min_samples_leaf=8, max_features=0.4, n_jobs=-1, random_state=3)
rf.fit(tr[feat_cols], tr['y'])
p = rf.predict_proba(te[feat_cols])[:,1]
final = {'auc': float(roc_auc_score(te['y'], p)),
         'acc': float(accuracy_score(te['y'],(p>0.5).astype(int))),
         'balacc': float(balanced_accuracy_score(te['y'],(p>0.5).astype(int))),
         'pr_auc': float(average_precision_score(te['y'],p)),
         'n': len(te),
         'confusion': confusion_matrix(te['y'],(p>0.5).astype(int)).tolist(),
         'up_rate': float(te['y'].mean()),
         'pred_up_rate': float((p>0.5).mean())}
cm = confusion_matrix(te['y'],(p>0.5).astype(int))
print(f"\nFINAL RF (train<=2024, test 2025-2026): AUC={final['auc']:.4f} ACC={final['acc']:.4f} PR-AUC={final['pr_auc']:.4f} n={final['n']}")
print(f"confusion (rows true: down/up, cols pred down/up):\n{cm}")

pi = permutation_importance(rf, te[feat_cols], te['y'], n_repeats=15, random_state=3, scoring='roc_auc', n_jobs=-1)
fi = pd.Series(pi.importances_mean, index=feat_cols).sort_values(ascending=False)
final['feature_importance'] = {k: round(float(v),4) for k,v in fi.items()}
with open(f"{OUT}\\ml_06.json","w") as fd: json.dump(df_out, fd, indent=2, default=str)

fig, ax = plt.subplots(figsize=(8,6))
fi.head(15).plot.barh(ax=ax, color='#1f77b4')
ax.set_title('Permutation importance (drop in ROC-AUC) — predicts next-day UP vs DOWN\n(random forest, test 2025–2026)')
ax.invert_yaxis()
fig.tight_layout(); fig.savefig(f"{OUT}\\c13_feat_importance.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- AUC by threshold confidence (mean-ret by decile) ----------
te2 = te.copy(); te2['prob'] = p; te2['actual_ret'] = te2['ret_next']
te2['decile'] = pd.qcut(te2['prob'], 10, labels=False)
dec = te2.groupby('decile').agg(n=('y','count'), pred_up=('prob','mean'), actual=('y','mean'), mean_ret=('actual_ret','mean'))
print("\nDecile of model probability vs realized next-day return (test 2025-2026):")
print(dec.round(4))
final['deciles'] = dec.round(4).to_dict('index')

fig, ax = plt.subplots(figsize=(8,4))
ax2 = ax.twinx()
ax.bar(dec.index, dec['actual']*100, color='#2ca02c', alpha=0.7, label='% up next day')
ax2.plot(dec.index, dec['mean_ret']*100, 'o-', color='#d62728', label='mean next-day ret %')
ax.set_title('Model confidence decile vs realized next-day return (test 2025–2026)')
ax.set_xlabel('Confidence decile (0=lowest P(up))'); ax.set_ylabel('% up next day'); ax2.set_ylabel('mean next-day ret %')
fig.tight_layout(); fig.savefig(f"{OUT}\\c14_deciles.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- bias / monotonicity ----------
final['decile_correlation'] = float(np.corrcoef(dec.index, dec['actual'])[0,1])
with open(f"{OUT}\\ml_06.json","w") as fd: json.dump(df_out, fd, indent=2, default=str)
with open(f"{OUT}\\ml_final_06.json","w") as fd: json.dump(final, fd, indent=2, default=str)


In [ ]:
# ---------- simple interpretability: SK learn LR coefficients ----------
lrf = Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=3000, C=0.5))])
lrf.fit(tr[feat_cols], tr['y'])
coef = pd.Series(lrf.named_steps['m'].coef_[0], index=feat_cols).sort_values()
final['logreg_top_coefs_down'] = coef.head(8).round(4).to_dict()
final['logreg_top_coefs_up'] = coef.tail(8).round(4).to_dict()
with open(f"{OUT}\\ml_final_06.json","w") as fd: json.dump(final, fd, indent=2, default=str)
print("\nLR top coefficients (up-drivers):\n", coef.tail(8))
print("LR top coefficients (down-drivers):\n", coef.head(8))

---

## 07 - Time-Series Decomposition & Forecasting

Decomposes monthly log price, extracts cycle peaks/troughs, and benchmarks naive/AR/ARIMA walk-forward forecasts. Outputs ts_07.json and c15.

In [ ]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('statsmodels').setLevel(logging.ERROR)
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from scipy.signal import argrelextrema
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

OUT = r"C:\Users\Admin\Downloads\archive\out"
m = pd.read_parquet(f"{OUT}\\btc_monthly.parquet")
m = m[m.index >= '2017-01-01']
res = {}
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox':'tight', 'axes.grid':True, 'grid.alpha':0.3,
                     'axes.spines.top':False, 'axes.spines.right':False, 'font.size':9})


In [ ]:
# ---- monthly log decomposition ----
ls = np.log(m['Close'])
dec = seasonal_decompose(ls, model='additive', period=12, extrapolate_trend='freq')
fig, ax = plt.subplots(4,1, figsize=(13,10), sharex=True)
ax[0].plot(ls.index, ls, color='#1f77b4'); ax[0].set_ylabel('log(Close)'); ax[0].set_title('Monthly log price decomposition')
ax[1].plot(dec.trend.index, dec.trend, color='#ff7f0e'); ax[1].set_ylabel('Trend')
ax[2].plot(dec.seasonal.index, dec.seasonal, color='#2ca02c'); ax[2].set_ylabel('Seasonal')
ax[3].plot(dec.resid.index, dec.resid, color='#d62728'); ax[3].set_ylabel('Residual')
for a in ax: a.xaxis.set_major_locator(mdates.YearLocator())
fig.tight_layout(); fig.savefig(f"{OUT}\\c15_decomp_monthly.png"); display(fig); plt.close(fig)
res['decomp'] = {
 'seasonal_amplitude_log': float(dec.seasonal.max()-dec.seasonal.min()),
 'resid_std_log': float(dec.resid.std()),
 'trend_slope_last_12m': float((dec.trend.iloc[-1]-dec.trend.iloc[-13])),
}

# peak / trough cycle identification on monthly log price
locs_max = argrelextrema(ls.values, np.greater_equal, order=3)[0]
locs_min = argrelextrema(ls.values, np.less_equal, order=3)[0]
res['peaks'] = [[str(ls.index[i].strftime('%Y-%m')), float(np.exp(ls.values[i]))] for i in locs_max]
res['troughs'] = [[str(ls.index[i].strftime('%Y-%m')), float(np.exp(ls.values[i]))] for i in locs_min]
print("Cycles (monthly log-price extrema):")
print("  peaks:", res['peaks'])
print("  troughs:", res['troughs'])


In [ ]:
# ---- daily point forecasting: naive vs AR vs ARIMA, walk-forward ----
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index); lpc = np.log(d['Close'])
eval_folds = [f"{y}-01-01" for y in range(2021, 2027)]
horizons = [1, 5, 21]
methods = {}
for nm, order in [('naive_last', None), ('ar1', (1,0,0)), ('arima_1_1_0', (1,1,0)), ('ar5', (5,0,0))]:
    hres = {h: {'rmse': [], 'mae': [], 'n': 0} for h in horizons}
    for fld in eval_folds:
        T = pd.Timestamp(fld)
        te = lpc[(lpc.index>=T) & (lpc.index< T+pd.DateOffset(years=1))]
        tr = lpc[lpc.index<T]
        if len(te) < 40: continue
        if nm=='naive_last':
            p0 = tr.iloc[-1]
            for h in horizons:
                tt = te.values[h-1::h][:len(te.values)//h]
                if len(tt)==0: tt = te.values[:1]
                pp = np.full(len(tt), p0)
                hres[h]['rmse'].append(np.sqrt(mean_squared_error(tt, pp)))
                hres[h]['mae'].append(mean_absolute_error(tt, pp)); hres[h]['n'] += len(tt)
        else:
            f = ARIMA(tr, order=order).fit()
            for h in horizons:
                if len(te)//h == 0:
                    tt = te.values[:1]; pp = np.asarray(f.forecast(1))
                else:
                    preds = f.forecast(len(te)+h)
                    tt = te.values[h-1::h][:len(te.values)//h]
                    pp = np.asarray(preds)[h-1::h][:len(tt)]
                hres[h]['rmse'].append(np.sqrt(mean_squared_error(tt, pp)))
                hres[h]['mae'].append(mean_absolute_error(tt, pp)); hres[h]['n'] += len(tt)
    methods[nm] = {str(h): {'rmse_log': float(np.sqrt(np.mean([x**2 for x in v['rmse']]))),
                            'mae_log': float(np.mean(v['mae']))} for h, v in hres.items()}
print("\nForecasting OOS RMSE on LOG price (2021-2026, walk-forward):")
for nm, mv in methods.items():
    print(f"  {nm:16s}", {h: f"{v['rmse_log']*100:.1f}%" for h,v in mv.items()}, "| approx price RMSE:", {h: f"${(np.expm1(v['rmse_log']))*lpc.iloc[-1]:,.0f}" for h,v in mv.items()})
res['forecast_rmse'] = methods


In [ ]:
# ---- return-level benchmarks (predict next-day return) ----
r = d['ret'].dropna().values
roll30 = pd.Series(r).rolling(30).mean().shift(1).values
mask = np.isfinite(roll30)
rmse_30m = np.sqrt(mean_squared_error(r[mask][1:], roll30[mask][:-1]))*100
print(f"\n30d-avg as next-day return forecast: RMSE={rmse_30m:.3f}% vs unconditional daily std {r.std()*100:.3f}% (skill ~0)")
res['return_forecast'] = {'rmse_30dmean_pct': float(rmse_30m), 'uncond_std_pct': float(r.std()*100)}

with open(f"{OUT}\\ts_07.json","w") as fd: json.dump(res, fd, indent=2, default=str)
print("\nsaved ts_07.json")

---

## 08 - Correlation Heatmap & Executive Dashboard

Spearman correlation heatmap of daily metrics and the executive dashboard composite. Produces c16-c17.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

OUT = r"C:\Users\Admin\Downloads\archive\out"
d = pd.read_parquet(f"{OUT}\\btc_daily.parquet")
d.index = pd.to_datetime(d.index)
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox':'tight', 'axes.grid':True, 'grid.alpha':0.3,
                     'axes.spines.top':False, 'axes.spines.right':False, 'font.size':9})


In [ ]:
# ---------- c16: correlation heatmap of key daily metrics ----------
cols = ['ret','logret','range_pct','Volume','vol_change','vol_30','sma30_ratio','sma200_ratio','gap','rsi-ish']
h = pd.DataFrame(index=d.index)
h['ret'] = d['ret']; h['logret'] = d['logret']
h['range%'] = d['range_pct']; h['Volume'] = d['Volume']; h['VolΔ'] = d['vol_change']
h['Vol30(ann)'] = d['vol_30']
h = h.replace([np.inf,-np.inf],np.nan)
c = h.corr(method='spearman')
fig, ax = plt.subplots(figsize=(8,6.5))
im = ax.imshow(c.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(c))); ax.set_yticks(range(len(c)))
ax.set_xticklabels(c.columns, rotation=45, ha='right'); ax.set_yticklabels(c.columns)
for i in range(len(c)):
    for j in range(len(c)):
        ax.text(j, i, f"{c.values[i,j]:.2f}", ha='center', va='center', fontsize=8,
                color='white' if abs(c.values[i,j])>0.6 else 'black')
ax.set_title('Spearman correlation of daily metrics')
fig.colorbar(im, shrink=0.8)
fig.tight_layout(); fig.savefig(f"{OUT}\\c16_corr_heat.png"); display(fig); plt.close(fig)


In [ ]:
# ---------- c17: Executive dashboard composite ----------
fig = plt.figure(figsize=(15,9))
gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.28)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(d.index, d['Close']/1000, lw=0.8, color='#1f77b4')
ax1.set_yscale('log'); ax1.set_ylabel('Close ($K, log)')
ax1.set_title('BTC/USD price (log) — 14.5 years of microstructure data')
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(d.groupby(d.index.year)['Volume'].sum().index.astype(str),
        d.groupby(d.index.year)['Volume'].sum().values, color='#8c564b', width=0.8)
ax2.tick_params(axis='x', rotation=45); ax2.set_title('Volume (BTC yr)')
ax3 = fig.add_subplot(gs[1, 1])
av = d.groupby(d.index.year)['ret'].std()*np.sqrt(365)*100
ax3.bar(av.index.astype(str), av.values, color='#ff7f0e', width=0.8)
ax3.tick_params(axis='x', rotation=45); ax3.set_title('Annualized vol %')
ax4 = fig.add_subplot(gs[1, 2])
ax4.hist(d['ret'].dropna()*100, bins=100, color='#2ca02c')
ax4.set_title('Daily returns % (fat tails)')
ax5 = fig.add_subplot(gs[2, 0])
yr = d.groupby(d.index.year)['ret'].sum()*100
ax5.bar(yr.index.astype(str), yr.values, color=['#d62728' if v<0 else '#2ca02c' for v in yr.values], width=0.8)
ax5.tick_params(axis='x', rotation=45); ax5.axhline(0,color='k',lw=0.6)
ax5.set_title('Yearly return % (green/red)')
ax6 = fig.add_subplot(gs[2, 1])
ax6.fill_between(d.index, d['maxdrawdown_hist']*100, 0, color='#d62728', alpha=0.7)
ax6.set_title('Drawdown % from ATH'); ax6.xaxis.set_major_locator(mdates.YearLocator())
ax7 = fig.add_subplot(gs[2, 2])
ax7.plot(d['vol_30'], lw=0.8, color='#9467bd')
ax7.set_title('30d realized vol (ann. %)'); ax7.xaxis.set_major_locator(mdates.YearLocator())
fig.suptitle('Executive Dashboard — Bitcoin 1-min to daily analytics', fontsize=13, y=0.99)
fig.savefig(f"{OUT}\\c17_dashboard.png"); display(fig); plt.close(fig)
print("c16, c17 saved")